# 03 - Traditional NLP baseline: predict CFPB Issue

This notebook compares two text-only baselines: TF-IDF + Logistic Regression and TF-IDF + Linear SVM. The target is the CFPB `Issue` column. Rows without a narrative or issue are excluded, then the remaining data is split into a stratified 80/20 train/test set with random seed 42. The TF-IDF vectorizer is inside each model pipeline, so vocabulary and IDF weights are learned from training narratives only. Both classifiers use `class_weight="balanced"`.

Macro-F1, macro precision, macro recall, one-vs-rest macro ROC-AUC, per-class reports, and labeled confusion matrices are shown below. The notebook identifies weak classes by test-set F1; do not interpret scores until the actual CFPB CSV has been supplied and the cells have run.

In [ ]:
# Import paths and display tools for loading and reviewing the model results.
from pathlib import Path
import sys

import pandas as pd

# Find the project root whether Jupyter starts in the root or notebooks folder.
project_root = Path.cwd()
if not (project_root / "src").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

# Import the reusable split, training, and reporting code.
from src.traditional_baseline import (
    DEFAULT_RANDOM_SEED,
    evaluate_baselines,
    print_results,
    write_results,
)

In [ ]:
# Read only the two modeling columns to avoid loading unrelated consumer fields.
input_csv = project_root / "data" / "raw" / "complaints.csv"
report_dir = project_root / "data" / "processed" / "baseline"

if input_csv.is_file():
    complaints = pd.read_csv(
        input_csv,
        usecols=["Consumer complaint narrative", "Issue"],
        encoding="utf-8-sig",
        dtype=str,
    )

    # The helper drops blank narratives/labels and performs the stratified holdout.
    baseline_results = evaluate_baselines(
        complaints,
        narrative_column="Consumer complaint narrative",
        target_column="Issue",
        test_size=0.2,
        random_seed=DEFAULT_RANDOM_SEED,
    )

    # Print full per-class reports and confusion matrices, then save reproducible outputs.
    print_results(baseline_results)
    write_results(baseline_results, report_dir)
    print(f"\nJSON and CSV reports saved under: {report_dir}")
else:
    print(f"CFPB CSV not found: {input_csv}. Add the source file and rerun this cell.")

## Interpreting the comparison

- **Macro-F1** gives every issue equal weight, making it useful when frequent issues could otherwise dominate the score. Macro precision measures correctness of class assignments on average; macro recall measures how many examples of each issue are recovered.
- **Macro one-vs-rest ROC-AUC** uses predicted probabilities for Logistic Regression and decision scores for Linear SVM. It measures ranking, not the selected class label. It is supplementary: a high ROC-AUC does not guarantee a high macro-F1.
- In each classification report, `support` is the number of held-out examples for that issue. Sort or review low F1 and low recall classes, but read support alongside the score: a class with very few test examples has an unstable estimate.
- In each confusion matrix, rows are actual issues and columns are predicted issues. Off-diagonal cells show which issues the model confuses. Frequent confusion between related descriptions can reflect overlapping issue language; poor performance in a small class can also reflect limited narrative examples.
- `weakest_classes` is computed from the current held-out per-class F1 values. It is the evidence-based list to discuss after running on the CFPB file, not a prewritten claim about which categories are weakest.

No CFPB CSV is included in the project by default, so this repository does not hard-code or invent evaluation scores. Run the notebook with the downloaded source to populate the results.